# 2. Hyperspectral FTH across the Co edge region

Hold the sample fixed and reload optical constants and wavelength at each energy. Produce both common-q complex reconstructions and a fixed physical-camera series. The sparse default energies illustrate the workflow; they are not a finely resolved spectrum.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/scattering_calculator').is_dir())
PAPER = ROOT / 'paper/scattering_calculator'
sys.path.insert(0, str(PAPER))
sys.path.insert(0, str(ROOT / "src"))
from scattering_calculator import simulation_pipelines as sim
import experiments as ex
import workflow as wf

def show(name):
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(plt.imread(OUT / (name + '.png')))
    ax.axis('off')
    plt.show()


## Editable configuration
Choose energies within the bundled Co magnetic-channel window (770–805 eV).

## Standard experiment setup

Use the same production configuration sections in every example:
**X-ray source → simulation grid → sample → magnetization → apertures →
illumination → sample propagation → detector/acquisition → beamstop → energy scan**.
Physical lengths are metres, energy is eV, and angles are radians; recipe
thickness literals are nm. See [the setup standard](../../../docs/experiment_setup.md).
Specialized plotting, reconstruction and analytic-reference settings are analysis
controls, separate from the physical experiment definition.


In [ ]:
OUT = PAPER / 'results' / 'notebook_hyperspectral'

# 1. X-RAY SOURCE — energy in eV, flux in photons/s, angles in radians
# 2. SIMULATION GRID — (y, x) pixels and physical pitch in metres
# 3. SAMPLE — recipe thickness literals are nm; slice thickness is metres
# 4. MAGNETIZATION — all physical pattern lengths are metres
# 5. APERTURES — positions are (y, x) in metres
# 6. ILLUMINATION — profile, focus and direction
# 7. SAMPLE PROPAGATION — field representation and multislice
# 8. DETECTOR / ACQUISITION — physical geometry, response and artifacts
# 9. BEAMSTOP; 10. OUTPUTS — scan axes live in illumination; analysis is separate
experiment = sim.ExperimentConfig(
    xray=sim.XRayConfig(energy=778., photon_flux=1., pol="CR",
                       coherence_length=(100e-6, 100e-6)),
    simulation=sim.SimulationConfig(shape=(192,192), real_space_pixel_size=10e-9),
    sample=sim.SampleConfig(recipe="Au(1000)/SiN(20)/Pt(2)/Co(10)/Pt(2)",
                            max_slice_thickness=5e-9, sample_name="Normal-incidence FTH"),
    magnetic_pattern=sim.MagneticPatternConfig(
        pattern_type_method="binary_labyrinth_pattern", pattern_config={"period":110e-9,"n_steps": 4}),
    aperture=sim.FrontApertureConfig(aperture_method="FTH_circular", aperture_config={
        "apertures_type":["OH","RH"], "apertures_radius":[180e-9,35e-9],
        "apertures_center":[(0.,0.),(0.,600e-9)], "apertures_sigma":[0.,0.],
        "apertures_top_radius_factor":[1.,1.], "thickness_OH":1000e-9,
    }),
    illumination=sim.IlluminationConfig(illumination_function="gaussian", illumination_config={
        "center":(0.,0.), "distance":0., "fwhm":2*np.sqrt(np.log(2))*650e-9,
        "alpha_beam":(0.,0.),
    }, energies_eV=tuple(np.arange(772.,805.,2.5)), polarizations=("CR", "CL")),
    propagation=sim.SamplePropagatorConfig(propagator_method="Jones", propagator_config={
        "propagate":True, "dielectric_tensor_compact":True,
    }),
    detector=sim.DetectorConfig(shape=(256,256), pixel_size=13.5e-6,
        sample_to_detector_distance=.05, detector_center=(128,128),
        detector_propagation_method="fraunhofer", use_detector_pixel_footprint=False,
        detector_pixel_footprint_samples=3, analyzer_angle=None,
        measurement_config={"exposure_time":1000.,"number_frames":10,"max_counts_per_image":10e3},
        detector_params={"counts_per_photon":100.,"quantum_efficiency":1.,
            "readout_noise_average":30.,"readout_noise_sigma":3.,"detector_threshold":16000.,
            "noise_seed":17},
        artifacts_config={"sigma_photon":0.,"camera_seed":31,"average_hot_pixels":0.,
                          "average_cold_pixels":0.,"cosmic_rays_per_second":0.}),
    beamstop=sim.BeamstopConfig(bs_method="circular",bs_detector_distance=.01,
                              bs_config={"radius":40.5e-6}),
    outputs=sim.OutputConfig(path=OUT / "experiment.h5", overwrite=True,
        save=("exit_wave", "fft_intensity", "detector_ideal", "detector_measured", "reconstruction")),
)


## Run, save, reload and plot
The runner reads the detector, scan axes and output choices from `experiment`. Each exposure is saved to one HDF5 file with its optical constants and camera q coordinates. An omitted scan list uses the single source energy/polarization. The measured image uses configured photon flux and exposure, without a synthetic peak count budget.


In [ ]:
# This same command handles one energy or an energy/polarization scan.
results = sim.simulate_experiment(experiment)
# Loading and plotting read only the saved HDF5 file.
results = sim.load_results(experiment.outputs.path)
fig = sim.plot_results(results, save_path=OUT / "experiment_summary.png")
plt.show()


## Camera pixels versus reciprocal coordinates
The fixed camera stays in place, but qx and qy change with energy. Regrid before common-q reconstruction. The saved common-q cube is calculated before physical detector projection. Its reference sideband is a finite-aperture correlation, not a calibrated magnetization inversion. The historical `sideband_rms` key stores root-sum-square amplitude over the ROI.

In [ ]:
common_reconstruction = results.stack('reconstruction')
fixed_images = results.stack('detector_ideal')
qx = results.stack('coordinates/qx')
print('Common reciprocal-grid reconstructions:', common_reconstruction.shape)
print('Fixed-detector images:', fixed_images.shape)
print('Per-exposure q grids:', qx.shape)
print('Energy (eV):', results.energies_eV)
print('Polarizations:', results.polarizations)

# CR/CL contrast is postprocessing of the saved data, not an extra simulation.
if results.polarizations == ('CR', 'CL'):
    contrast = common_reconstruction[:, 0] - common_reconstruction[:, 1]
    signal = np.sqrt(np.sum(abs(contrast)**2, axis=(1,2)))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    axes[0].plot(results.energies_eV, signal, 'o-')
    axes[0].set(xlabel='energy (eV)', ylabel='CR−CL correlation norm')
    axes[1].imshow(abs(contrast[0]), cmap='magma')
    axes[1].set_title(f'CR−CL correlation at {results.energies_eV[0]:g} eV')
    plt.show()
